In [ ]:

prompt_templates = """
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>
{think}
</think>
{answer}<|im_end|>
"""

In [ ]:
data = [
    {"question": "你是谁", "think": "我是谁，我要思考一下，我是一个学习助手", "answer": "我是你的学习助手"},
    {"question": "你是谁开发的", "think": "我要思考一下，我是由慧科团队开发的", "answer": "我是由慧科团队开发的"},
    {"question": "你能做什么", "think": "我要思考一下，我能为你解答学习问题", "answer": "我能为你解答学习问题"},
]

In [ ]:
for item in data:
    print(prompt_templates.format(question=item["question"], think=item["think"], answer=item["answer"]))


In [ ]:
from modelscope import AutoModelForCausalLM, AutoTokenizer

model_name = "openai-community/gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.add_special_tokens({"bos_token": "<|im_start|>"})
tokenizer.add_special_tokens({"eos_token": "<|im_end|>"})
tokenizer.add_special_tokens({"pad_token": "<|endoftext|>"})

model.resize_token_embeddings(len(tokenizer))


In [ ]:
from torch.utils.data import Dataset


class dataset(Dataset):
    def __init__(self, data, max_length=256):
        self.encodings = []
        for item in data:
            text = prompt_templates.format(question=item["question"], think=item["think"], answer=item["answer"])
            encoded = tokenizer(
                text,
                max_length=max_length,
                truncation=True,
                padding='max_length',
                return_tensors="pt"
            )
            input_ids = encoded['input_ids'].squeeze()
            self.encodings.append(input_ids)

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


dataset = dataset(data)
dataset[0]

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

dataCollator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./model",
    per_device_train_batch_size=1,
    num_train_epochs=100,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=dataCollator
)

trainer.train()
trainer.save_model("../models")

In [ ]:
chat_templates = """
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>

"""
prompt = "你能做什么"

text = chat_templates.format(question=prompt)
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=256,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id)
content = tokenizer.decode(generated_ids[0])
print(content)